# 可选实验: 模型表示

<figure>
 <img src="../work/images/C1_W1_L3_S1_Lecture_b.png"   style="width: 700px;height:200px;">
</figure>

## 目标
在这个实验你将:
- 学习实现单变量线性回归模型 $f_{w,b}$ 。

## 符号
以下是您将遇到的部分符号的总结。

|通用符号  | 描述 | Python (if applicable) |
|:-- |:-- |:-- |
| $a$ | 标量，非粗体           |
| $\mathbf{a}$ | 向量，粗体                                                      |
| **回归** |         |    
|  $\mathbf{x}$ | 训练样本特征值（在本实验中为房屋面积（1000平方英尺））  | `x_train` |   
|  $\mathbf{y}$  | 训练样本目标值（在本实验中为价格（千美元）） | `y_train` 
|  $x^{(i)}$, $y^{(i)}$ | $i_{th}$个训练样本 | `x_i`, `y_i`|
| m | 训练样本数量 | `m`|
|  $w$  |  参数：权重,                                 | `w`    |
|  $b$           |  参数：偏置                                          | `b`    |     
| $f_{w,b}(x^{(i)})$ | 模型在参数$w,b$ 下对 $x^{(i)}$ 的评估结果: $f_{w,b}(x^{(i)}) = wx^{(i)}+b$  | `f_wb` | 



## 工具
在本实验中，你将使用: 
- NumPy, 一个流行的科学计算库
- Matplotlib, 一个流行的数据绘图库

In [2]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')

# 问题描述
<img align="left" src="../work/images/C1_W1_L3_S1_trainingdata.png"    style=" width:500px; padding: 10px;  " /> 

与讲座中一样，你将使用房价预测作为激励示例。
本实验将使用一个仅包含两个数据点的简单数据集：一栋1000平方英尺（sqft）的房屋以*300,000*美元售出，一栋2000平方英尺的房屋以*500,000*美元售出。这两个点将构成我们的*数据或训练集*。在本实验中，面积的单位是1000平方英尺，价格的单位是千美元。

| 面积 (1000平方英尺)     | 价格 (千美元) |
| -------------------| ------------------------ |
| 1.0               | 300                      |
| 2.0               | 500                      |

你希望通过这两个点拟合一个线性回归模型（如上图蓝色直线所示），以便随后预测其他房屋的价格——例如，一栋1200平方英尺的房屋。


请运行以下代码单元以创建你的 `x_train` 和 `y_train` 变量. 数据存储在一维 NumPy 数组中。

In [3]:
# x_train is the input variable (size in 1000 square feet)
# y_train is the target (price in 1000s of dollars)
x_train = np.array([1.0, 2.0])
y_train = np.array([300.0, 500.0])
print(f"x_train = {x_train}")
print(f"y_train = {y_train}")

x_train = [1. 2.]
y_train = [300. 500.]


>**注意**: 本课程在打印时会频繁使用 Python 的 'f-string' 输出格式化，其描述见 [此处](https://docs.python.org/3/tutorial/inputoutput.html)。花括号内的内容在生成输出时会被求值。

### 训练样本数 `m`
你将使用 m 来表示训练样本的数量。NumPy 数组有一个 `.shape` 属性。`x_train.shape` 返回一个 Python 元组，其中包含每个维度的条目。`x_train.shape[0]` 是数组的长度，也就是样本数量，如下所示：

In [5]:
# m is the number of training examples
print(f"x_train.shape: {x_train.shape}")
m = x_train.shape[0]
print(f"Number of training examples is: {m}")

你也可以使用Python的`len()`函数，如下所示。

In [6]:
# m is the number of training examples
m = len(x_train)
print(f"Number of training examples is: {m}")

### 训练样本 `x_i, y_i`

你将使用 (x$^{(i)}$, y$^{(i)}$) 来表示第 $i^{th}$ 个训练样本。 由于 Python 使用零索引， (x$^{(0)}$, y$^{(0)}$) 是 (1.0, 300.0) 而 (x$^{(1)}$, y$^{(1)}$) 是 (2.0, 500.0)。

要访问 NumPy 数组中的值，需要使用所需的偏移量对数组进行索引。例如，访问`x_train` 中位置零的语法是 `x_train[0]`。   

运行下面的下一个代码块以获取第 $i^{th}$ 个训练样本。

In [7]:
i = 0 # Change this to 1 to see (x^1, y^1)

x_i = x_train[i]
y_i = y_train[i]
print(f"(x^({i}), y^({i})) = ({x_i}, {y_i})")

### 绘制数据

你可以使用 `matplotlib` 库中的 `scatter()` 函数绘制这两个点，如下面的单元格所示。
- 函数参数 `marker` 和 `c` 将点显示为红色叉号（默认为蓝色圆点）。

你可以使用 `matplotlib` 库中的其他函数来设置要显示的标题和标签。

In [8]:
# 绘制数据点图
plt.scatter(x_train, y_train, marker='x', c='r')
# 设置标题
plt.title("Housing Prices")
# 设置 y-axis 标签
plt.ylabel('Price (in 1000s of dollars)')
# 设置 x-axis 标签
plt.xlabel('Size (1000 sqft)')
plt.show()

## 模型函数

<img align="left" src="../work/images/C1_W1_L3_S1_model.png"     style=" width:600px; padding: 10px; " >

如讲座中所述, 线性回归的模型函数（一个从 x 映射到 y 的函数）表示为

$$ f_{w,b}(x^{(i)}) = wx^{(i)} + b \tag{1}$$

上面的公式展示了如何表示直线——不同的 $w$ 和 $b$ 值会在图上给出不同的直线。

让我们通过下面的代码块来更好地理解这一点。让我们从 $w=100$ 和 $b=100$ 开始。

<br/>
<br/>
<br/>
<br/>
<br/>
<br/>

**注意: 你可以返回这个单元格来调整模型的 w 和 b 参数**

In [9]:
w = 50
b = 200
print(f"w: {w}")
print(f"b: {b}")

现在，让我们为你的两个数据点计算 $f_{w,b}(x^{(i)})$ 的值. 你可以为每个数据点显式地写出如下：

对于 $x^{(0)}$, `f_wb = w * x[0] + b`

对于 $x^{(1)}$, `f_wb = w * x[1] + b`

对于大量数据点，这可能会变得繁琐且重复。因此，你可以改为在 `for` 循环中计算函数输出，如下面 `compute_model_output` 函数所示。

> **注意**: 参数描述 `(ndarray (m,))` 描述了一个形状为 (m,) 的 NumPy n 维数组，`(scalar)` 描述了一个没有维度的参数，仅表示大小。

> **注意**: `np.zero(n)`将返回一个包含 $n$ 个条目的一维 NumPy 数组。


In [10]:
def compute_model_output(x, w, b):
    """
    Computes the prediction of a linear model
    Args:
      x (ndarray (m,)): Data, m examples 
      w,b (scalar)    : model parameters  
    Returns
      y (ndarray (m,)): target values
    """
    m = x.shape[0]
    f_wb = np.zeros(m)
    for i in range(m):
        f_wb[i] = w * x[i] + b
        
    return f_wb

现在让我们调用 `compute_model_output` 函数并绘制输出...

In [11]:
tmp_f_wb = compute_model_output(x_train, w, b,)

# Plot our model prediction
plt.plot(x_train, tmp_f_wb, c='b',label='Our Prediction')

# Plot the data points
plt.scatter(x_train, y_train, marker='x', c='r',label='Actual Values')

# Set the title
plt.title("Housing Prices")
# Set the y-axis label
plt.ylabel('Price (in 1000s of dollars)')
# Set the x-axis label
plt.xlabel('Size (1000 sqft)')
plt.legend()
plt.show()

如你所见, 设置 $w = 100$ and $b = 100$ 不会产生一条拟合我们数据的直线。

### 挑战
尝试用不同的 $w$ 和 $b$ 值进行实验。一条拟合我们数据的直线，其值应该是多少？

#### 提示:
你可以用鼠标点击下方绿色"提示"左侧的三角形，以显示一些选择 b 和 w 的提示。

<details>
<summary>
    <font size='3', color='darkgreen'><b>查看提示</b></font>
</summary>
    <p>
    <ul>
        <li>Try $w = 200$ and $b = 100$ </li>
    </ul>
    </p>

### 预测
现在我们有了一个模型，我们可以用它来进行最初的预测。让我们预测一栋1200平方英尺房屋的价格。由于 $x$ 的单位是1000平方英尺，所以 $x$ 是1.2。


In [ ]:
w = 200                         
b = 100    
x_i = 1.2
cost_1200sqft = w * x_i + b    

print(f"${cost_1200sqft:.0f} thousand dollars")

# 祝贺你!
在这个实验室里你学会了:
 - 线性回归构建一个模型，该模型建立了特征与目标之间的关系
     - 在上面的例子中，特征是房屋面积，目标是房屋价格。
     - 对于简单线性回归，模型有两个参数 $w$ 和 $b$，其值使用训练数据进行*拟合*。
     - 一旦模型的参数被确定，该模型就可以用于对新数据进行预测。